In [28]:
#!pip install transformers datasets torch torchvision torchaudio
#!pip install sentencepiece rouge-score

In [29]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import BartTokenizer
import re

In [30]:
!pip install huggingface

In [32]:
#!pip install torch transformers datasets rouge-score

## Load the CNN Dataset

In [33]:
from datasets import load_dataset

'''# First time: download and save to disk
dataset = load_dataset("cnn_dailymail", "3.0.0")
dataset.save_to_disk("./cnn_dailymail_local")  # saves as Arrow format'''


'# First time: download and save to disk\ndataset = load_dataset("cnn_dailymail", "3.0.0")\ndataset.save_to_disk("./cnn_dailymail_local")  # saves as Arrow format'

In [34]:

# Every run after → load instantly from disk (no internet needed)
from datasets import load_from_disk
dataset = load_from_disk("./cnn_dailymail_local")

In [35]:
dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [36]:
print(dataset['train'][0]['article'][:300]) 
print(" ") 
print(dataset['train'][0]['highlights'])

LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappoi
 
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


## Preprocess & Tokenize

In [37]:
import re

def clean_text(text):
    text = text.lower()                          # lowercase everything
    text = re.sub(r'\s+', ' ', text)             # remove extra spaces
    text = re.sub(r'[^a-zA-Z0-9\s.,]', '', text) # remove special chars
    return text.strip()

In [38]:
# Apply cleaning
train_articles   = [clean_text(x['article'])    for x in dataset['train'].select(range(10000))]
train_summaries  = [clean_text(x['highlights']) for x in dataset['train'].select(range(10000))]

### We use 10000 samples only — full dataset is too large for college project

In [39]:
print(f"Total samples: {len(train_articles)}")
print(f"Sample article: {train_articles[0][:200]}")
print(f"Sample summary: {train_summaries[0]}")

Total samples: 10000
Sample article: london, england reuters  harry potter star daniel radcliffe gains access to a reported 20 million 41.1 million fortune as he turns 18 on monday, but he insists the money wont cast a spell on him. dani
Sample summary: harry potter star daniel radcliffe gets 20m fortune as he turns 18 monday . young actor says he has no plans to fritter his cash away . radcliffes earnings from first five potter films have been held in trust fund .


## Build Vocabulary
### The model doesn't understand words — it understands numbers. So we convert every word to a number.

In [40]:
from collections import Counter

In [41]:

# Special tokens every seq2seq model needs
PAD_TOKEN = '<PAD>'   # padding (makes all sentences same length)
SOS_TOKEN = '<SOS>'   # start of summary
EOS_TOKEN = '<EOS>'   # end of summary
UNK_TOKEN = '<UNK>'   # unknown word

In [42]:
def build_vocab(texts, max_vocab=20000):
    counter = Counter()
    for text in texts:
        counter.update(text.split())
    
    # Start vocab with special tokens
    vocab = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2, UNK_TOKEN: 3}
    
    # Add most common words
    for word, _ in counter.most_common(max_vocab):
        vocab[word] = len(vocab)
    
    return vocab

In [43]:

# Build vocab from articles + summaries combined
all_text = train_articles + train_summaries
vocab = build_vocab(all_text)

# Reverse vocab (number → word) for decoding output later
idx2word = {v: k for k, v in vocab.items()}

print(f"Vocabulary size: {len(vocab)}")
print(f"Word 'patient' is number: {vocab.get('patient', 3)}")

Vocabulary size: 20004
Word 'patient' is number: 2472


##  Convert Text to Numbers

In [44]:
import torch
from torch.utils.data import Dataset, DataLoader

In [45]:

MAX_ARTICLE_LEN = 400   # max words in input
MAX_SUMMARY_LEN = 80    # max words in output

In [46]:
def text_to_tensor(text, vocab, max_len):
    words = text.split()[:max_len]
    # Convert word → number, use UNK if word not in vocab
    ids = [vocab.get(word, vocab[UNK_TOKEN]) for word in words]
    # Pad to max_len
    ids += [vocab[PAD_TOKEN]] * (max_len - len(ids))
    return torch.tensor(ids, dtype=torch.long)

class SummarizationDataset(Dataset):
    def __init__(self, articles, summaries, vocab):
        self.articles   = articles
        self.summaries  = summaries
        self.vocab      = vocab

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        src = text_to_tensor(self.articles[idx],  self.vocab, MAX_ARTICLE_LEN)
        tgt = text_to_tensor(self.summaries[idx], self.vocab, MAX_SUMMARY_LEN)
        return src, tgt

# Create dataset and dataloader
train_dataset = SummarizationDataset(train_articles, train_summaries, vocab)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"Number of batches: {len(train_loader)}")


Number of batches: 313


##  Build the LSTM Model

In [47]:
import torch.nn as nn

# ── ENCODER ──────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        
        # Embedding: converts word numbers → vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # LSTM layer: reads sequence and produces hidden states
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers,
                            batch_first=True, dropout=dropout)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src shape: [batch, src_len]
        embedded = self.dropout(self.embedding(src))
        # outputs: all hidden states at each word
        # hidden, cell: final memory (what encoder "remembers")
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell



In [48]:

# ── ATTENTION ────────────────────────────────────────
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v    = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: [batch, hidden_dim]
        # encoder_outputs: [batch, src_len, hidden_dim]
        
        src_len = encoder_outputs.shape[1]
        
        # Repeat hidden state for each source word
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        
        # Calculate attention scores
        energy = torch.tanh(self.attn(
            torch.cat((hidden, encoder_outputs), dim=2)
        ))
        attention = self.v(energy).squeeze(2)
        
        # Softmax → attention weights (sum to 1)
        return torch.softmax(attention, dim=1)


In [49]:


# ── DECODER ──────────────────────────────────────────
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = Attention(hidden_dim)
        
        # Input = embedding + context vector from attention
        self.lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim, n_layers,
                            batch_first=True, dropout=dropout)
        
        # Final layer: hidden state → word probability
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt_word, hidden, cell, encoder_outputs):
        # tgt_word: [batch] — one word at a time
        tgt_word = tgt_word.unsqueeze(1)                      # [batch, 1]
        embedded  = self.dropout(self.embedding(tgt_word))    # [batch, 1, embed]

        # Attention
        attn_weights    = self.attention(hidden[-1], encoder_outputs)
        attn_weights    = attn_weights.unsqueeze(1)           # [batch, 1, src_len]
        context         = torch.bmm(attn_weights, encoder_outputs)  # [batch, 1, hidden]

        # Combine embedding + context
        lstm_input      = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        prediction = self.fc_out(output.squeeze(1))           # [batch, vocab_size]
        return prediction, hidden, cell



In [50]:

# ── SEQ2SEQ (puts encoder + decoder together) ────────
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, vocab, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.vocab   = vocab
        self.device  = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size  = src.shape[0]
        tgt_len     = tgt.shape[1]
        vocab_size  = len(self.vocab)

        # Store decoder outputs
        outputs = torch.zeros(batch_size, tgt_len, vocab_size).to(self.device)

        # Encode input
        encoder_outputs, hidden, cell = self.encoder(src)

        # First decoder input = <SOS> token
        dec_input = torch.full((batch_size,), self.vocab[SOS_TOKEN],
                               dtype=torch.long).to(self.device)

        for t in range(tgt_len):
            pred, hidden, cell = self.decoder(dec_input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = pred

            # Teacher forcing: sometimes use real word, sometimes use predicted word
            use_teacher = torch.rand(1).item() < teacher_forcing_ratio
            dec_input = tgt[:, t] if use_teacher else pred.argmax(1)

        return outputs

## Train the Model

In [51]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.inference_mode():
        for src, tgt in loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            output = model(src, tgt, teacher_forcing_ratio=0)  # no teacher forcing
            output = output[:, 1:, :].reshape(-1, VOCAB_SIZE)
            tgt    = tgt[:, 1:].reshape(-1)
            total_loss += criterion(output, tgt).item()
    return total_loss / len(loader)

# Validation dataloader
val_articles  = [clean_text(x['article'])    for x in dataset['validation'].select(range(2000))]
val_summaries = [clean_text(x['highlights']) for x in dataset['validation'].select(range(2000))]
val_dataset   = SummarizationDataset(val_articles, val_summaries, vocab)
val_loader    = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [52]:
from torch.cuda.amp import GradScaler, autocast  # ← ADD THIS

In [53]:
from torch.cuda.amp import GradScaler, autocast        # enables fp16 mixed precision training

# ── Model Hyperparameters ─────────────────────────────────────────────────────
VOCAB_SIZE  = len(vocab)       # total unique words in our vocabulary
EMBED_DIM   = 128              # size of word embedding vectors
HIDDEN_DIM  = 256              # size of LSTM hidden state
N_LAYERS    = 2                # number of stacked LSTM layers
DROPOUT     = 0.3              # randomly drops 30% of neurons to prevent overfitting
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # use GPU if available

# ── Build Model ───────────────────────────────────────────────────────────────
encoder = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)  # reads the article
decoder = Decoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)  # generates the summary
model   = Seq2Seq(encoder, decoder, vocab, DEVICE).to(DEVICE)             # combines both, moves to GPU

# ── Optimizer, Loss, Scheduler ────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
# Adam optimizer — adjusts learning rate per parameter automatically
# lr=0.0005 is safer than 0.001 — less chance of overshooting

criterion = nn.CrossEntropyLoss(ignore_index=vocab[PAD_TOKEN])
# CrossEntropyLoss — measures how wrong the predictions are
# ignore_index=PAD — we don't penalize the model for padding tokens

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=1)
# automatically reduces learning rate when val_loss stops improving
# patience=1 means it waits 1 epoch before reducing LR

scaler = GradScaler()
# GradScaler works with autocast() to keep fp16 training numerically stable
# it scales up gradients to prevent them becoming zero in fp16

# ── Training Function ─────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()       # sets model to training mode (enables dropout, batch norm)
    total_loss = 0

    for batch_idx, (src, tgt) in enumerate(loader):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)   # move data to GPU

        optimizer.zero_grad(set_to_none=True)
        # clears gradients from previous batch
        # set_to_none=True is faster than filling with zeros

        with autocast():
            # autocast automatically runs operations in fp16 where safe
            # this gives ~2x speedup with no accuracy loss
            output = model(src, tgt)                              # forward pass → [batch, tgt_len, vocab_size]
            output = output[:, 1:, :].reshape(-1, VOCAB_SIZE)    # skip first token, flatten for loss
            tgt    = tgt[:, 1:].reshape(-1)                      # skip <SOS> token in target
            loss   = criterion(output, tgt)                       # calculate how wrong predictions are

        scaler.scale(loss).backward()
        # scales loss up before backprop so fp16 gradients don't vanish
        # then computes gradients for all parameters

        scaler.unscale_(optimizer)
        # IMPORTANT: unscale gradients back to real values BEFORE clipping
        # without this, clipping operates on inflated values and never triggers

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        # gradient clipping — if gradients get too large (exploding gradients)
        # this caps them at 1.0 to keep training stable

        scaler.step(optimizer)
        # unscales gradients (if not done already) and calls optimizer.step()
        # skips the step if gradients contain inf/nan (fp16 safety)

        scaler.update()
        # updates the scale factor for next iteration
        # increases scale if no inf/nan, decreases if there were

        total_loss += loss.item()   # accumulate loss for epoch average

        if batch_idx % 50 == 0:
            print(f"  Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(loader)   # return average loss across all batches

# ── Run Training ──────────────────────────────────────────────────────────────
N_EPOCHS = 5
for epoch in range(N_EPOCHS):
    print(f"\nEpoch {epoch+1}/{N_EPOCHS}")

    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    # trains on all 10000 samples, returns average loss

    val_loss = evaluate(model, val_loader, criterion)
    # runs on validation set with no gradient updates
    # teacher_forcing_ratio=0 inside evaluate() — model uses its own predictions

    scheduler.step(val_loss)
    # checks if val_loss improved — if not, reduces learning rate
    # helps escape plateaus in training

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    # watching both losses:
    # if train_loss drops but val_loss rises → overfitting
    # if both drop together → good training

# ── Save Model ────────────────────────────────────────────────────────────────
torch.save({
    'model_state': model.state_dict(),  # all learned weights
    'vocab':       vocab,               # word → number mapping (needed for inference)
    'idx2word':    idx2word,            # number → word mapping (needed to decode output)
}, 'lstm_summarizer.pt')

print("Model saved!")

C:\Users\manis\AppData\Local\Temp\ipykernel_18640\3633342869.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
c:\Users\manis\anaconda3\envs\medirepo\lib\site-packages\torch\cuda\amp\grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
C:\Users\manis\AppData\Local\Temp\ipykernel_18640\3633342869.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
c:\Users\manis\anaconda3\envs\medirepo\lib\site-packages\torch\cuda\amp\autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(



Epoch 1/5
  Batch 0/313 | Loss: 9.9058
  Batch 50/313 | Loss: 7.3458
  Batch 100/313 | Loss: 7.1154
  Batch 150/313 | Loss: 7.1189
  Batch 200/313 | Loss: 7.1879
  Batch 250/313 | Loss: 7.0844
  Batch 300/313 | Loss: 7.1328
Train Loss: 7.3014 | Val Loss: 6.8240

Epoch 2/5
  Batch 0/313 | Loss: 7.1083
  Batch 50/313 | Loss: 7.2464
  Batch 100/313 | Loss: 6.9795
  Batch 150/313 | Loss: 7.0777
  Batch 200/313 | Loss: 7.0512
  Batch 250/313 | Loss: 7.0024
  Batch 300/313 | Loss: 7.0819
Train Loss: 7.0527 | Val Loss: 6.8121

Epoch 3/5
  Batch 0/313 | Loss: 7.0545
  Batch 50/313 | Loss: 7.0774
  Batch 100/313 | Loss: 7.0407
  Batch 150/313 | Loss: 6.9761
  Batch 200/313 | Loss: 6.8887
  Batch 250/313 | Loss: 6.9028
  Batch 300/313 | Loss: 6.9670
Train Loss: 6.9945 | Val Loss: 6.8274

Epoch 4/5
  Batch 0/313 | Loss: 6.9796
  Batch 50/313 | Loss: 6.8754
  Batch 100/313 | Loss: 6.8640
  Batch 150/313 | Loss: 6.9670
  Batch 200/313 | Loss: 6.8896
  Batch 250/313 | Loss: 6.9202
  Batch 300/313 |

## Generate a Summary (Test It)

In [56]:
def generate_summary(model, article, vocab, idx2word, max_len=80):
    model.eval()
    with torch.no_grad():
        src = text_to_tensor(clean_text(article), vocab, MAX_ARTICLE_LEN)
        src = src.unsqueeze(0).to(DEVICE)  # add batch dimension

        encoder_outputs, hidden, cell = model.encoder(src)
        dec_input = torch.tensor([vocab[SOS_TOKEN]]).to(DEVICE)

        summary_words = []
        for _ in range(max_len):
            pred, hidden, cell = model.decoder(dec_input, hidden, cell, encoder_outputs)
            top_word = pred.argmax(1)

            word = idx2word.get(top_word.item(), UNK_TOKEN)
            if word == EOS_TOKEN:
                break
            summary_words.append(word)
            dec_input = top_word

    return ' '.join(summary_words)

# Test it
test_article = """
The patient is a 45 year old male who was admitted with severe chest pain 
and difficulty breathing. Tests revealed he has type 2 diabetes and 
hypertension. He was prescribed metformin 500mg and lisinopril 10mg daily.
"""

summary = generate_summary(model, test_article, vocab, idx2word)
print(f"Generated Summary:\n{summary}")

Generated Summary:
<UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> . . . . <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> . . . . <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .


In [55]:
from rouge_score import rouge_scorer

def evaluate_rouge(model, articles, summaries, vocab, idx2word, n=100):
    scorer  = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores  = {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}

    for article, ref_summary in zip(articles[:n], summaries[:n]):
        gen = generate_summary(model, article, vocab, idx2word)
        s   = scorer.score(ref_summary, gen)
        scores['rouge1'] += s['rouge1'].fmeasure
        scores['rouge2'] += s['rouge2'].fmeasure
        scores['rougeL'] += s['rougeL'].fmeasure

    print(f"ROUGE-1: {scores['rouge1']/n:.4f}")
    print(f"ROUGE-2: {scores['rouge2']/n:.4f}")
    print(f"ROUGE-L: {scores['rougeL']/n:.4f}")

evaluate_rouge(model, val_articles, val_summaries, vocab, idx2word)

ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000


## What Each Part Does — Quick Reference

| Part              | Why it exists                                      |
|------------------|----------------------------------------------------|
| Encoder          | Reads input article, creates memory                |
| Attention        | Decides which words in input matter most           |
| Decoder          | Uses memory to write summary word by word          |
| Teacher Forcing  | Training trick — helps model learn faster          |
| Vocab            | Converts words ↔ numbers                           |
| PAD token        | Makes all sentences same length for batching       |